## PART THREE OF PYTHON

# Understanding and Trusting Data
Data doesn't lie, but it can hide. When a government audit threatens to freeze the Ministry of Agriculture's funding, the gap between an unverified spreadsheet and a defensible dataset is three weeks of rigorous proof. In this project, you step into an emergency data audit for Maji Ndogo: you profile distributions, break rainfall down by province, weigh each crop's yield, and trace the true drivers of productivity through a Seaborn correlation heatmap. It is exactly the kind of high-stakes, story-driven exploratory data analysis that proves you can trust data under pressure.

The letter from the government auditor arrives on a Tuesday.

It is polite but unmistakable: the Ministry of Agriculture has three weeks to demonstrate that the Maji Ndogo farm survey data is accurate before it can be used to allocate funding. If the data cannot be independently verified, the entire project gets put on hold.

Sanaa reads the letter twice, puts it face-down on the desk, and turns to you.

"We have weather stations across the region," she says. "Five of them. They record temperature, rainfall, and pollution independently of our field teams — IoT sensors, real-time data. If our field measurements agree with the station readings, we have a defensible case." She pauses. "If they do not..."

She does not finish the sentence.

Your job starts now. The audit clock is running, and the first step is to understand the data as it stands — distribution by distribution, relationship by relationship, pattern by pattern. This notebook is your field report. Every chart you produce, every number you note down, is evidence you can take into that audit. Work carefully.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text

engine = create_engine('sqlite:///Maji_Ndogo_farm_survey_small.db')
sql_query = """
SELECT *
FROM geographic_features
LEFT JOIN weather_features USING (Field_ID)
LEFT JOIN soil_and_crop_features USING (Field_ID)
LEFT JOIN farm_management_features USING (Field_ID)
"""
with engine.connect() as connection:
    MD_agric_df = pd.read_sql_query(text(sql_query), connection)

# In this survey, the Crop_type and Annual_yield columns are stored in each
# other's place. Swap them back so Crop_type holds crop names and Annual_yield
# holds the numeric tonnage.
MD_agric_df.rename(columns={'Annual_yield': 'Crop_type_Temp', 'Crop_type': 'Annual_yield'}, inplace=True)
MD_agric_df.rename(columns={'Crop_type_Temp': 'Crop_type'}, inplace=True)

# A handful of elevations were logged as negative — fix the sign.
MD_agric_df['Elevation'] = MD_agric_df['Elevation'].abs()

# Clean known crop-name typos (e.g. 'cassaval' -> 'cassava').
corrections = {'cassaval': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'}
MD_agric_df['Crop_type'] = MD_agric_df['Crop_type'].apply(
    lambda crop: corrections.get(str(crop).strip(), str(crop).strip())
)
print(f'Loaded: {MD_agric_df.shape[0]} rows x {MD_agric_df.shape[1]} columns')

Loaded: 5654 rows x 18 columns


### Data Dictionary

**Geographic features**

| Column | Description | Type |
|--------|-------------|------|
| Field_ID | Unique identifier for each field | BigInt |
| Elevation | Elevation above sea level in meters | Float |
| Latitude / Longitude | Geographic coordinates in degrees | Float |
| Location | Province the field is in | Text |
| Slope | Slope of the land in degrees | Float |

**Weather features**

| Column | Description | Type |
|--------|-------------|------|
| Rainfall | Annual rainfall in mm | Float |
| Min_temperature_C | Average minimum temperature in degrees C | Float |
| Max_temperature_C | Average maximum temperature in degrees C | Float |
| Ave_temps | Average temperature in degrees C | Float |

**Soil and crop features**

| Column | Description | Type |
|--------|-------------|------|
| Soil_fertility | Normalized fertility index, 0-1 | Float |
| Soil_type | Categorical soil classification | Text |
| pH | Soil pH level | Float |
| Pollution_level | Normalized pollution index, 0-1 | Float |

**Farm management features**

| Column | Description | Type |
|--------|-------------|------|
| Plot_size | Field area in hectares | Float |
| Crop_type | Type of crop grown | Text |
| Annual_yield | Total annual yield in tonnes | Float |
| Standard_yield | Normalized yield score, 0-1 | Float |

## Challenge 1.1: Dataset Summary Metrics

Every analysis starts by understanding what you are working with. Before building complex charts, let's extract the essential dimensions: how many columns the joined survey has, and the average elevation across all fields.

### Task
Write a function `get_dataset_summary` that accepts the dataset and returns a tuple `(number_of_columns, mean_elevation)`, where `number_of_columns` is an `int` and `mean_elevation` is the mean of the `Elevation` column.

> ⚠️ Do not change the function name `get_dataset_summary`.

### Expected Output
```
(18, 637.7907086314113)
```

In [2]:
def get_dataset_summary(df):
    summary = {
        'num_columns': df.shape[1],
        'mean_elevation': df['Elevation'].mean(),
    }
    return summary

In [3]:
print(get_dataset_summary(MD_agric_df))

{'num_columns': 18, 'mean_elevation': np.float64(637.7907086314113)}


## Challenge 2.1: Rainfall by Location

The overall `Rainfall` distribution shows multiple peaks — a sign that different sub-populations are being averaged together. Breaking it down by province reveals the hidden structure, and tells the audit which regions are genuinely dry and which are wet.

### Task
Write a function `get_mean_rainfall_by_location` that calculates the mean `Rainfall` grouped by `Location` and returns it as a pandas Series sorted in **ascending** order of values.

> ⚠️ Do not change the function name `get_mean_rainfall_by_location`.

In [8]:
def get_mean_rainfall_by_location(df):
    return df.groupby('Location')['Rainfall'].mean().sort_values(ascending=True)

In [9]:
print(get_mean_rainfall_by_location(MD_agric_df))

Location
Rural_Amanzi       723.628958
Rural_Kilimani     791.352426
Rural_Hawassa     1325.941003
Rural_Akatsi      1584.884457
Rural_Sokoto      1705.079431
Name: Rainfall, dtype: float64


Amanzi and Kilimani are the dry provinces; Sokoto receives more than twice their rainfall. That spread is exactly what a per-location KDE plot would show as separated peaks.

## Challenge 3.1: Crop Distribution Volume

Before diving into continuous relationships, it helps to understand the categorical landscape. How many fields are planted with each crop type? This Series drives both a bar chart and a pie chart.

### Task
Write a function `get_crop_counts` that counts the number of fields per `Crop_type` and returns a pandas Series sorted in **descending** order of counts.

> ⚠️ Do not change the function name `get_crop_counts`.

In [13]:
def get_crop_counts(df):
    return df['Crop_type'].value_counts().sort_values(ascending=False)

In [14]:
get_crop_counts(MD_agric_df)

Crop_type
wheat      1342
tea         913
potato      823
cassava     672
banana      633
coffee      607
maize       399
rice        265
Name: count, dtype: int64

Wheat dominates the survey; rice is rare. An imbalance like this matters later — any model trained on this data will see wheat five times as often as rice.

## Challenge 4.1: Crop Median Yields

A violin plot shows the full distribution of a continuous variable across categories. Let's isolate the precise median annual yields across crop varieties to locate the performance peaks.

### Task
Write a function `get_median_yield_by_crop` that calculates the median `Annual_yield` grouped by `Crop_type` and returns a pandas Series sorted in **descending** order of median yield.

> ⚠️ Do not change the function name `get_median_yield_by_crop`.


In [15]:
def get_median_yield_by_crop(df):
    return df.groupby('Crop_type')['Annual_yield'].median().sort_values(ascending=False)

In [16]:
print(get_median_yield_by_crop(MD_agric_df))

Crop_type
rice       1.734914
tea        1.657955
cassava    1.447799
maize      1.446741
coffee     1.405154
banana     1.391763
wheat      1.389082
potato     1.283974
Name: Annual_yield, dtype: float64


Rice is rare (Challenge 3) but its median yield is the highest — a pattern worth flagging for the audit, since low-volume, high-yield crops are easy to overlook in an averaged view.

## Challenge 5.1: Temperature and Elevation Correlation

Scatter plots reveal how two continuous variables relate. Let's quantify the linear connection between altitude and average ambient temperature with a single Pearson coefficient.

### Task
Write a function `get_elevation_temp_correlation` that computes the Pearson correlation coefficient between `Elevation` and `Ave_temps`, returned as a `float`.

> ⚠️ Do not change the function name `get_elevation_temp_correlation`.

In [19]:

def get_elevation_temp_correlation(df):
    return df['Elevation'].corr(df['Ave_temps'])


In [20]:
get_elevation_temp_correlation(MD_agric_df)

np.float64(0.20306114718387713)

A weak positive correlation. One might expect higher elevation to mean *cooler* temperatures, so a mild positive value is itself an audit flag — the kind of "does this make physical sense?" check that validating data against independent sensors is meant to catch.

## Challenge 6.1: Yield Correlations

A correlation matrix measures how strongly each numerical variable relates to every other. Values close to 1 or -1 indicate a strong linear relationship. Here we ask: what actually drives `Standard_yield`?

### Task
Write a function `get_yield_correlations` that:
- Considers these feature columns only: `['Elevation', 'Slope', 'Rainfall', 'Min_temperature_C', 'Max_temperature_C', 'Ave_temps', 'Soil_fertility', 'pH', 'Pollution_level', 'Plot_size', 'Annual_yield']`.
- Computes each feature's Pearson correlation with `Standard_yield`.
- Returns the result as a pandas Series sorted in **descending order of absolute value** (strongest relationship first).

> ⚠️ Do not change the function name `get_yield_correlations`.

In [40]:
def get_yield_correlations(df):
    feature_columns = ['Elevation', 'Slope', 'Rainfall', 'Min_temperature_C', 'Max_temperature_C', 'Ave_temps', 'Soil_fertility', 'pH', 'Pollution_level', 'Plot_size', 'Annual_yield']
    df_subset = df[feature_columns + ['Standard_yield']]
    return df_subset.corr()['Standard_yield'].sort_values(ascending=False)
    

In [41]:
print(get_yield_correlations(MD_agric_df))

Standard_yield       1.000000
Annual_yield         0.220812
Min_temperature_C    0.144233
Elevation            0.129248
Soil_fertility       0.070205
Slope                0.056991
Rainfall             0.039217
Ave_temps            0.006786
Plot_size           -0.017014
Max_temperature_C   -0.111649
pH                  -0.196613
Pollution_level     -0.285761
Name: Standard_yield, dtype: float64


`Pollution_level` is the strongest single correlate of standardized yield (negative — more pollution, lower yield), ahead of soil and weather. None of the relationships is especially strong, which tells the audit that yield is multi-causal: no single field measurement explains it, so the data must be trusted as a whole, not by any one column.

## part two

# Validating Maji Ndogo's Statistical Consistency

Guessing a tolerance window isn’t science; it’s a liability. When an audit hinges on whether a 2% variance is a data flaw or natural noise, guessing leaves a budget exposed. In this project, you will use inferential statistics to defend Maji Ndogo's data. You will calculate sample metrics with Bessel's correction, map the Central Limit Theorem, construct 95% confidence intervals, and write an automated one-sample t-test script. It is exactly the kind of rigorous statistical proof required to ensure data credibility for international agricultural audits.

Sanaa calls you in at 7 a.m. She has been up since four.

"I have been thinking about that comparison," she says. "True: 7, False: 8. The tolerance I used was 1.5%. But why 1.5%? I made that number up." She draws a circle on the whiteboard. "What if the measurements are naturally noisy? What if a 2% difference between a sensor and a field reading is completely normal given how spread out the data is? We would be flagging perfectly good data as a problem."

She looks at you. "We need to stop guessing and start using statistics."

That is the shift this week. Instead of asking "is this within a tolerance window I invented?", you will ask the question the scientific way: *Could the difference between our field measurements and the weather station readings be explained by random variation alone?*

The tools you will use — population metrics, the Central Limit Theorem, confidence intervals, and hypothesis testing — form the backbone of inferential statistics. By the end of this notebook, you will have a defensible, statistically grounded answer for the auditor.

In [43]:
weather_station_mapping_df = pd.read_csv(
    'Weather_data_field_mapping.csv'
)
MD_agric_df = MD_agric_df.merge(weather_station_mapping_df, on='Field_ID', how='left')
MD_agric_df = MD_agric_df.drop(columns='Unnamed: 0', errors='ignore')
print(f'Loaded: {MD_agric_df.shape[0]} rows x {MD_agric_df.shape[1]} columns')

Loaded: 5654 rows x 19 columns


## Challenge 1.2: Sample vs. population metrics

Our 5,654 field records are the **population** — every farm in the survey. Any subset we analyze is a **sample**. The variance formula changes depending on whether you are describing the population or estimating it from a sample: population variance divides by `n`; sample variance divides by `n-1` (Bessel's correction, which removes bias in the estimate).

### Task

1. Calculate the population mean and variance of `Standard_yield` using `ddof=0`.
2. Call `np.random.seed(42)` on the first line of the cell, then draw a random
   sample of 50 fields using the `.sample()` method.
3. Calculate the sample mean and variance of that sample using `ddof=1`.
4. Print all four values.

In [50]:
np.random.seed(42)
sample_df = MD_agric_df.sample(n=50, random_state=42)
print(f"population mean: {MD_agric_df['Standard_yield'].mean():.4f}")
print(f"population variance: {MD_agric_df['Standard_yield'].var():.6f}")
print(f"sample mean: {sample_df['Standard_yield'].mean():.4f}")
print(f"sample variance: {sample_df['Standard_yield'].var():.6f}")

population mean: 0.5344
population variance: 0.012501
sample mean: 0.5305
sample variance: 0.010990


## Challenge 2.2: The Central Limit Theorem

A single sample mean wobbles around the true population mean. The Central Limit Theorem (CLT) says: take enough samples, and the distribution of those sample means will be approximately normal — centered on the population mean — regardless of what the original data looks like.

This is why we can use statistical tests on real-world data that is not normally distributed: the test statistics are based on sample means, not the raw data.

### Task

1. Call `np.random.seed(42)` on the first line of the cell, then draw 1,000 samples of 30 fields each using the `.sample()` method
2. Compute the mean `Standard_yield` of each sample and store results in a list called `sample_means`.
3. Plot a histogram of `sample_means` with a title and axis labels.
4. Print the mean of the sample means, their standard deviation, and the theoretical standard error (`population_std / sqrt(30)`).

### Expected output

```
Mean of sample means: 0.5343
Std of sample means: 0.0207
Theoretical standard error: 0.0204
```

In [59]:
np.random.seed(42)
sample_df = MD_agric_df.sample(n=1000, random_state=42)
print(f"Mean of sample means: {sample_df['Standard_yield'].mean():.4f}")
print(f"Std of sample means: {sample_df['Standard_yield'].std():.4f}")
print(f"Theoretical standard error: {MD_agric_df['Standard_yield'].std() / np.sqrt(30):.4f}")

Mean of sample means: 0.5411
Std of sample means: 0.1166
Theoretical standard error: 0.0204


## Challenge 3: Confidence intervals

A **95% confidence interval** gives a range: if we repeated this survey many times and built a CI each time, 95% of those intervals would contain the true population mean. It does **not** mean there is a 95% probability that the true mean lies in this particular interval — the true mean either is or is not in there.

SciPy syntax: `stats.t.interval(confidence, df, loc=mean, scale=sem)` — where `sem` is `stats.sem(data)` and `df` is `len(data) - 1`.

### Task

1. Calculate the 95% CI for the mean `Standard_yield` across all 5,654 fields.
2. Calculate the 95% CI for `Rainfall` in fields assigned to **weather station 0** only.

In [64]:
from scipy import stats

sem = stats.sem(MD_agric_df['Standard_yield'])
yield_ci = stats.t.interval(
    0.95,
    len(MD_agric_df['Standard_yield']) - 1,
    loc=MD_agric_df['Standard_yield'].mean(),
    scale=sem
)

station_0_rainfall = MD_agric_df.loc[
    MD_agric_df['Weather_station'] == 0, 'Rainfall'
]
station_0_n = len(station_0_rainfall)
station_0_mean = station_0_rainfall.mean()
station_0_sem = stats.sem(station_0_rainfall)
station_0_ci = stats.t.interval(
    0.95,
    station_0_n - 1,
    loc=station_0_mean,
    scale=station_0_sem
)

print(
    f"Mean yield: {MD_agric_df['Standard_yield'].mean():.4f}, "
    f"95% CI: ({yield_ci[0]:.4f}, {yield_ci[1]:.4f})"
)
print(
    f"n: {station_0_n}, Mean rainfall: {station_0_mean:.2f} mm, "
    f"95% CI: ({station_0_ci[0]:.2f}, {station_0_ci[1]:.2f})"
)
# The published mean of 1,575.95 mm falls outside the station 0 confidence interval.

Mean yield: 0.5344, 95% CI: (0.5315, 0.5373)
n: 1375, Mean rainfall: 1522.89 mm, 95% CI: (1499.83, 1545.96)


## Challenge 4.2: One-sample t-test

The confidence interval showed the station's published rainfall mean sits outside our field data's 95% CI. But a CI only tells us the range for our mean — it does not directly test whether the station value is plausible. For that, we use a hypothesis test.

**H₀:** The true mean rainfall near station 0 equals 1,575.95 mm — any difference is random noise.
**H₁:** The true mean is different from 1,575.95 mm.

We reject H₀ if the p-value is below α = 0.05. A small p-value means: "if H₀ were true, the probability of seeing data this extreme by chance is less than 5%."

### Task

1. Run `stats.ttest_1samp()` on the `Rainfall` data for fields near station 0, tested against the reference value `1575.95`. Print the t-statistic, p-value (5 decimal places), and your conclusion.
2. Repeat for **average temperature** (`Ave_temps`), tested against the station reference mean of `13.4039`. Print those results too.

In [68]:
station_0_temp = MD_agric_df.loc[
    MD_agric_df['Weather_station'] == 0, 'Ave_temps'
]

rainfall_result = stats.ttest_1samp(station_0_rainfall, 1575.95)
temperature_result = stats.ttest_1samp(station_0_temp, 13.4039)

print(
    f"Rainfall: T-statistic: {rainfall_result.statistic:.5f}, "
    f"P-value: {rainfall_result.pvalue:.5f}"
)
print(
    "Reject H₀ — rainfall discrepancy is statistically significant."
    if rainfall_result.pvalue < 0.05
    else "Fail to reject H₀ — rainfall difference is consistent with random noise."
)

print(
    f"Temperature: T-statistic: {temperature_result.statistic:.5f}, "
    f"P-value: {temperature_result.pvalue:.5f}"
)
print(
    "Reject H₀ — temperature discrepancy is statistically significant."
    if temperature_result.pvalue < 0.05
    else "Fail to reject H₀ — temperature difference is consistent with random noise."
)

Rainfall: T-statistic: -4.51238, P-value: 0.00001
Reject H₀ — rainfall discrepancy is statistically significant.
Temperature: T-statistic: -0.48144, P-value: 0.63028
Fail to reject H₀ — temperature difference is consistent with random noise.


## Challenge 5.2: All stations — systematic testing

Station 0's rainfall is a problem. But what about the other four stations? Running the test one station at a time is tedious and error-prone. We need a function that systematizes the process.

### Task

Complete `rainfall_ttest_results(df, station_means, alpha=0.05)`. The function must:
- Accept a DataFrame, a dictionary of `{station_id: reference_rainfall_mean}`, and a significance level.
- Loop over every station in `station_means`.
- Run a one-sample t-test on the field `Rainfall` for each station against its reference mean.
- Print one line per station showing the station ID, p-value, and whether H₀ was rejected.
- Include a docstring.

> ⚠️ Do not change the function name `rainfall_ttest_results`.

### Expected output

```
Significant difference in Rainfall at Station 0 (P-value: 0.00001 ≤ 0.05). H₀ rejected.
No significant difference in Rainfall at Station 1 (P-value: 0.06741 > 0.05). H₀ not rejected.
Significant difference in Rainfall at Station 2 (P-value: 0.00542 ≤ 0.05). H₀ rejected.
Significant difference in Rainfall at Station 3 (P-value: 0.00549 ≤ 0.05). H₀ rejected.
Significant difference in Rainfall at Station 4 (P-value: 0.00064 ≤ 0.05). H₀ rejected.
```

In [73]:
station_rainfall_means = {
    0: 1575.953750,
    1: 577.383910,
    2: 1690.955324,
    3: 905.191397,
    4: 1200.183505,
}

### START FUNCTION
def rainfall_ttest_results(df, station_means, alpha=0.05):
    results = {}
    for station, mean in station_means.items():
        station_rainfall = df.loc[df['Weather_station'] == station, 'Rainfall']
        t_statistic, p_value = stats.ttest_1samp(station_rainfall, mean)
        rejected = p_value < alpha
        results[station] = {
            't_statistic': t_statistic,
            'p_value': p_value,
            'rejected': rejected
        }
        if rejected:
            print(f"Station {station}, {t_statistic:.5f}, {p_value:.5f}: Reject H₀ — rainfall discrepancy is statistically significant.")
        else:
            print(f"Station {station}, {t_statistic:.5f}, {p_value:.5f}: Fail to reject H₀ — rainfall difference is consistent with random noise."),
            
print(rainfall_ttest_results(MD_agric_df, station_rainfall_means))

Station 0, -4.51270, 0.00001: Reject H₀ — rainfall discrepancy is statistically significant.
Station 1, -1.83105, 0.06741: Fail to reject H₀ — rainfall difference is consistent with random noise.
Station 2, 2.78702, 0.00542: Reject H₀ — rainfall discrepancy is statistically significant.
Station 3, 2.78171, 0.00549: Reject H₀ — rainfall discrepancy is statistically significant.
Station 4, -3.42208, 0.00064: Reject H₀ — rainfall discrepancy is statistically significant.
None


## part three

# Building the Maji Ndogo Data Pipeline

Notebooks are great for exploration, but production systems need something more resilient. When an audit deadline forces three consultants to work across drifting copies of the same analysis, messy copy-pasted code quickly becomes a maintenance nightmare. In this project you will refactor raw analytical code into a production-quality data pipeline for Maji Ndogo. You will work with modular Python scripts for SQL ingestion, run purpose-built data-processing classes, configure a single-source-of-truth parameter dictionary, and deploy automated two-sample Welch's t-tests to validate field measurements against weather station readings. This is the software-engineering transition that turns exploration scripts into reliable, reusable systems.

Sanaa sends an email at 11 p.m.

"I tried to run the analysis on the updated database and got an error on line 47. The column rename code is in three different notebooks, each slightly different. One of them has the old database filename. I fixed it in one place and it broke in another."

She adds: "We are not building a notebook. We are building a system. Notebooks are for exploration. Systems are for production. We need to separate the two."

This is the wall every data scientist hits eventually. Notebook code works, but it does not *scale*. When the same thirty lines of setup code appear in every notebook, slightly different each time, maintenance becomes a full-time job. The auditor's deadline made it worse — three people editing three notebooks in parallel, each drifting in a different direction.

The solution is **modular Python scripts**. One module for data ingestion. One for field data processing. One for weather data processing. Every notebook imports these modules in two lines. When something changes, you fix the module once and every notebook inherits the fix.

By the end of this project you will have a production-quality data pipeline: documented, testable, and reusable.


In [74]:
import numpy as np
import pandas as pd
import logging
from scipy.stats import ttest_ind

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

## Challenge 1: Document the data ingestion module

`data_ingestion.py` contains three functions that form the foundation of the pipeline: `create_db_engine` connects to the SQLite database, `query_data` runs a SQL query and returns a DataFrame, and `read_from_web_CSV` fetches a CSV from a URL. Each function is already implemented and includes error handling.

Good documentation is not optional in a production pipeline — it is what makes a script reusable by teammates, reviewable by auditors, and maintainable six months from now.

### Task

1. Open `data_ingestion.py`.
2. Add a **module-level docstring** at the very top of the file (before any imports) describing what the module does in one to three sentences.
3. Add a **PEP 257 compliant function docstring** to each of the three functions. Each docstring must include:
   - A one-line summary of what the function does.
   - A `Parameters` section listing every argument with its type and description.
   - A `Returns` section describing the return value and its type.
   - A `Raises` section listing any exceptions the function may raise.
4. Run the verification cell below.

> ⚠️ Do not change the function logic — only add documentation.

> **AI assist:** Paste each function signature and body into an AI assistant and ask: *"Generate a PEP 257 compliant docstring for this function — include Parameters, Returns, and Raises sections."* Then review and adapt the output before pasting it in.

### Expected output

```
data_ingestion
data_ingestion
data_ingestion
```

In [91]:
from data_ingestion import create_db_engine, query_data, read_from_web_CSV

print(create_db_engine.__module__)
print(query_data.__module__)
print(read_from_web_CSV.__module__)
### END FUNCTION

data_ingestion
data_ingestion
data_ingestion


## Challenge 2: Configure the pipeline

A production pipeline has one single source of truth for its parameters. Rather than hunting through the code every time the database path changes or a new crop spelling correction is discovered, every parameter lives in a single configuration dictionary. Change one value, and the whole pipeline picks it up.

The two weather-data URLs are already provided in the template below.

### Task

Complete the `config_params` dictionary. Every key is provided — fill in the four missing values using the parameters that have been consistent across all three notebooks in this course.

- `sql_query`: the four-table LEFT JOIN query used since Week 1.
- `db_path`: the SQLite connection string used since Week 1.
- `columns_to_rename`: the dictionary that corrects the swapped `Annual_yield` / `Crop_type` columns.
- `values_to_rename`: the dictionary of crop spelling corrections.

> ⚠️ Do not change the dictionary key names.

In [90]:
config_params = {
    'sql_query': """
    SELECT *
    FROM geographic_features
    LEFT JOIN weather_features USING (Field_ID)
    LEFT JOIN soil_and_crop_features USING (Field_ID)
    LEFT JOIN farm_management_features USING (Field_ID)
    """, 
    'db_path': 'sqlite:///Maji_Ndogo_farm_survey_small.db', 
    'columns_to_rename': {'Annual_yield': 'Crop_type_Temp', 'Crop_type': 'Annual_yield'}, 
    'values_to_rename': {'Crop_type_Temp': 'Crop_type'},
    'weather_csv_path': 'weather_station_data.csv', 
    'weather_mapping_csv': 'weather_data_field_mapping.csv'}

## Challenge 3: Run the field data processor

`FieldDataProcessor` in `field_data_processor.py` chains four processing steps into a single `.process()` call: it ingests the SQL data, corrects the swapped column names, applies crop-type spelling fixes, and merges the weather station mapping. All the logic is already implemented — your job is to document it so that anyone can understand it without reading the source code.

### Task

1. Open `field_data_processor.py`.
2. Add a **class-level docstring** to `FieldDataProcessor` describing its purpose, the `config_params` it expects, and the attribute it exposes (`self.df`).
3. Add a **method-level docstring** to every public method that does not already have one. At minimum: `__init__`, `ingest_sql_data`, `rename_columns`, `apply_corrections`, and `process`.
4. Run the verification cell below.

> ⚠️ Do not change the processing logic — only add documentation.

### Expected output

```
Shape: (5654, 19)
```

In [92]:
from field_data_processor import FieldDataProcessor

field_processor = FieldDataProcessor(config_params)
field_processor.process()

print(f"Shape: {field_processor.df.shape}")


2026-11-02 13:41:36,663 - data_ingestion - INFO - Database engine created successfully.
2026-11-02 13:41:36,748 - data_ingestion - INFO - Query executed successfully.
2026-11-02 13:41:36,749 - field_data_processor.FieldDataProcessor - INFO - Sucessfully loaded data.
2026-11-02 13:41:36,751 - field_data_processor.FieldDataProcessor - INFO - Swapped columns: Annual_yield with Crop_type_Temp


AttributeError: 'float' object has no attribute 'strip'

## Challenge 4.3: Run the weather data processor

`WeatherDataProcessor` in `weather_data_processor.py` loads the IoT sensor CSV and extracts structured measurements (Temperature, Rainfall, Pollution_level) from free-text messages. Document it the same way you documented the field processor.

### Task

1. Open `weather_data_processor.py`.
2. Add a **class-level docstring** and a **method-level docstring** to any method that does not already have one.
3. Run the verification cell below.

> ⚠️ Do not change the processing logic — only add documentation.

### Expected output

```
Shape: (1843, 4)
```

In [98]:
from weather_data_processor import WeatherDataProcessor

weather_processor = WeatherDataProcessor(config_params)
weather_processor.process()

print(f"Shape: {weather_processor.df.shape()}")


2026-11-02 13:43:17,534 - data_ingestion - INFO - CSV file read successfully from the web.
2026-11-02 13:43:17,536 - weather_data_processor.WeatherDataProcessor - INFO - Successfully loaded weather station data from the web.
2026-11-02 13:43:17,555 - weather_data_processor.WeatherDataProcessor - INFO - Messages processed and measurements extracted.


2026-11-02 13:43:17,556 - weather_data_processor.WeatherDataProcessor - INFO - Data processing completed.


AttributeError: 'WeatherDataProcessor' object has no attribute 'df'

## Challenge 5: Data validation with hypothesis testing

The pipeline is running and both DataFrames are loaded. Now we close the loop on the audit question: do the field measurements and weather station readings agree?

In Week 2 we used one-sample t-tests to compare field means against fixed reference values. Here we use **independent-samples t-tests** (`ttest_ind`) because we now have actual measurement samples from both sources — we are comparing two distributions, not one distribution against a constant.

Note that the field DataFrame uses the column name `Ave_temps` for temperature, while the weather DataFrame labels it `Temperature`. Your `filter_field_data` function must map between these names.

### Task

Complete the four functions below. Each must include a docstring and must not be hard-coded.

- `filter_field_data(df, station_id, measurement)` — return the corresponding column from the field DataFrame as a Series for rows where `Weather_station == station_id`, dropping NaN values. Use an internal mapping to handle the `Temperature` → `Ave_temps` name difference.
- `filter_weather_data(df, station_id, measurement)` — return the `Value` column as a Series for rows where `Weather_station_ID == station_id` and `Measurement == measurement`, dropping NaN values.
- `run_ttest(column_a, column_b)` — return `(t_stat, p_val)` from `ttest_ind` with `equal_var=False` and `alternative='two-sided'`.
- `print_ttest_results(station_id, measurement, p_val, alpha)` — print one clearly formatted result line per test. If `p_val > alpha`, indicate that H₀ is not rejected; otherwise indicate that H₀ is rejected.

> ⚠️ Do not change the function names.
No significant difference in Temperature at Station 0 (P-value: 0.90761 > 0.05). H₀ not rejected.
No significant difference in Rainfall at Station 0 (P-value: 0.21621 > 0.05). H₀ not rejected.
No significant difference in Pollution_level at Station 0 (P-value: 0.56418 > 0.05). H₀ not rejected.
No significant difference in Temperature at Station 1 (P-value: 0.47241 > 0.05). H₀ not rejected.

In [ ]:
def filter_field_data(df, station_id, measurement):
    """Filter field data for a specific weather station and measurement.

    Parameters
    ----------
    df : pandas.DataFrame
        The processed field DataFrame.
    station_id : int
        The weather station ID to filter on.
    measurement : str
        The measurement type (e.g., 'Ave_temps', 'Rainfall', 'Pollution_level').

    Returns
    -------
    pandas.Series
        A Series of measurement values with NaN values dropped.
    """
    pass


def filter_weather_data(df, station_id, measurement):
    """Filter weather data for a specific weather station and measurement.

    Parameters
    ----------
    df : pandas.DataFrame
        The processed weather DataFrame with Weather_station_ID, Measurement,
        and Value columns.
    station_id : int
        The weather station ID to filter on.
    measurement : str
        The measurement type (e.g., 'Temperature', 'Rainfall', 'Pollution_level').

    Returns
    -------
    pandas.Series
        A Series of measurement values with NaN values dropped.
    """
    pass


def run_ttest(column_a, column_b):
    """Run an independent two-sample Welch's t-test.

    Parameters
    ----------
    column_a : array-like
        First sample of observations.
    column_b : array-like
        Second sample of observations.

    Returns
    -------
    tuple
        A tuple of (t_statistic, p_value).
    """
    pass


def print_ttest_results(station_id, measurement, p_val, alpha):
    """Print a formatted result line for a single t-test comparison.

    Parameters
    ----------
    station_id : int
        The weather station ID being tested.
    measurement : str
        The measurement name being compared.
    p_val : float
        The p-value from the t-test.
    alpha : float
        The significance level.
    """
    pass

In [ ]:
measurements_to_compare = ['Temperature', 'Rainfall', 'Pollution_level']
alpha = 0.05

for station_id in sorted(weather_df['Weather_station_ID'].unique()):
    for measurement in measurements_to_compare:
        field_values   = filter_field_data(field_df, station_id, measurement)
        weather_values = filter_weather_data(weather_df, station_id, measurement)
        t_stat, p_val  = run_ttest(field_values, weather_values)
        print_ttest_results(station_id, measurement, p_val, alpha)